In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import constraints
from torch.func import vmap

In [5]:
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, TraceEnum_ELBO, config_enumerate, infer_discrete, Predictive, Importance, EmpiricalMarginal
from pyro.infer.mcmc.mcmc_kernel import MCMCKernel
from pyro.ops.indexing import Vindex
from pyro import poutine
from pyro.poutine import trace, replay

In [6]:
import numpy as np
import math
from itertools import product, accumulate
import einops
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import seaborn as sns
import umap  

In [7]:
def generalized_einsum(tensor_a, tensor_b):
    # Get shapes
    shape_a = tensor_a.shape  # e.g., (n, a1, a2, a3, a4, ...)
    shape_b = tensor_b.shape  # e.g., (..., k) = (a4, a3, a2, a1, k)
    
    # Extract n (first dimension of A)
    n = shape_a[0]
    
    # Extract additional dimensions from A (excluding n)
    dims_a = list(shape_a[1:])  # e.g., [a1, a2, a3, a4]
    
    # Extract dimensions from B (excluding k)
    dims_b = list(shape_b[:-1])  # e.g., [a4, a3, a2, a1]
    k = shape_b[-1]
    
    # Validate that B's dimensions (excluding k) are the reverse of A's (excluding n)
    if dims_a != dims_b[::-1]:
        raise ValueError(f"Dimensions {dims_a} and {dims_b} are not compatible (must be reverse order)")
    
    # Create dimension labels
    # Use unique letters for each dimension (e.g., i, j, k, l for a1, a2, a3, a4)
    dim_labels = [chr(105 + i) for i in range(len(dims_a))]  # e.g., ['i', 'j', 'k', 'l']
    
    # Pattern for A: n followed by additional dims
    pattern_a = f"a {' '.join(dim_labels)}"  # e.g., "n i j k l"
    
    # Pattern for B: reversed additional dims followed by k
    pattern_b = f"{' '.join(dim_labels[::-1])} z"  # e.g., "l k j i k"
    
    # Output pattern: n k
    pattern_out = "a z"
    
    # Full einsum pattern
    pattern = f"{pattern_a}, {pattern_b} -> {pattern_out}"  # e.g., "n i j k l, l k j i k -> n k"
    
    # Perform einsum
    return einops.einsum(tensor_a, tensor_b, pattern)

In [8]:
def mix_weights(beta):
    # Compute cumulative product of (1 - beta) along the last dimension
    beta1m_cumprod = (1 - beta).cumprod(dim=-1)
    # Pad beta with a 1 at the end of the last dimension
    beta_padded = F.pad(beta, (0, 1), value=1)
    # Pad beta1m_cumprod with a 1 at the start of the last dimension
    beta1m_cumprod_padded = F.pad(beta1m_cumprod, (1, 0), value=1)
    # Element-wise multiplication
    weight = beta_padded * beta1m_cumprod_padded
    rlt = torch.max(weight, torch.tensor(1e-6, device=beta.device))
    rlt = rlt/rlt.sum(dim=-1, keepdim=True)
    # return F.softmax(beta_padded * beta1m_cumprod_padded, dim=-1)
    return rlt
    

In [ ]:
def model(data, struct_upbd, vocab_size, device):

    param_dims = list(struct_upbd.values())
    param_dims.reverse()
    
    # model parameters
    struct_params = {}
    struct_params["gamma"] = pyro.param("model_gamma", torch.rand(1, device=device), constraint=constraints.positive)
    
    for parent_level in range(len(struct_upbd)-1):  
        child_level = parent_level + 1
        struct_params[f"alpha{parent_level}"] = pyro.param(f"model_alpha{parent_level}", torch.rand(param_dims[-child_level:-1], device=device) , constraint=constraints.positive).unsqueeze(-1).expand(param_dims[-child_level:])
        assert struct_params[f"alpha{parent_level}"].shape == tuple(param_dims[-child_level:])

        struct_params[f"eta{parent_level}"] = pyro.param(f"model_eta{parent_level}", torch.rand(param_dims[-child_level:-1], device=device), constraint=constraints.positive)
        assert struct_params[f"eta{parent_level}"].shape == tuple(param_dims[-child_level:-1])
    struct_params[f"alpha{len(struct_upbd)-1}"] = pyro.param(f"model_alpha{len(struct_upbd)-1}", torch.rand(param_dims[-len(struct_upbd)+1:-1], device=device), constraint=constraints.positive).unsqueeze(-1).expand(param_dims[-len(struct_upbd)+1:])

    struct_params["reg_mu_mu"] = pyro.param("model_reg_mu_mu", torch.zeros([struct_upbd["G0"],], device=device))
    struct_params["reg_mu_sigma"] = pyro.param("model_reg_mu_sigma", torch.ones([struct_upbd["G0"],], device=device), constraint=constraints.positive)
    struct_params["reg_sigma"] = pyro.param("model_reg_sigma", torch.ones([struct_upbd["G0"],], device=device), constraint=constraints.positive)

    # model structures
    struct_weights = {}
    struct_weights["B0"] = (torch.ones(1, device=device).expand(param_dims[-1:]), struct_params["gamma"].expand(param_dims[-1:]))
    beta_0 = pyro.sample("G0", dist.Beta(struct_weights["B0"][0], struct_weights["B0"][1]).to_event(1))
    assert beta_0.shape == (param_dims[-1], )

    struct_weights["G0"] = mix_weights(beta_0)[..., :-1]
    assert struct_weights["G0"].shape == (param_dims[-1], )

    for parent_level in range(len(struct_upbd)-1):
        child_level = parent_level + 1
        full_dim = child_level + 1

        param_alpha = struct_params[f"alpha{parent_level}"]*struct_weights[f"G{parent_level}"]
        param_beta = struct_params[f"alpha{parent_level}"]*(1 - struct_weights[f"G{parent_level}"].cumsum(-1))
        struct_weights[f"B{child_level}"] = (param_alpha.unsqueeze(0).expand(param_dims[-full_dim:]), param_beta.unsqueeze(0).expand(param_dims[-full_dim:]))
        beta = pyro.sample(f"G{child_level}", dist.Beta(struct_weights[f"B{child_level}"][0], struct_weights[f"B{child_level}"][1]).to_event(child_level))
        struct_weights[f"G{child_level}"] = mix_weights(beta)[..., :-1]
        assert struct_weights[f"G{child_level}"].shape == tuple(param_dims[-full_dim:])

    cluster_weights = {}
    for parent_level in range(len(struct_upbd)-1):
        child_level = parent_level + 1
        full_dim = child_level + 1
        beta = pyro.sample(f"L{parent_level}", dist.Beta(torch.ones_like(struct_params[f"eta{parent_level}"], device=device).unsqueeze(0).expand(param_dims[-full_dim:-1]), struct_params[f"eta{parent_level}"].unsqueeze(0).expand(param_dims[-full_dim:-1])).to_event(child_level))
        assert beta.shape == tuple(param_dims[-full_dim:-1])
        cluster_weights[f"L{parent_level}"] = mix_weights(beta)[..., :-1]

    mixture_components = {}
    mixture_components["generation"] = pyro.sample(f"gen", dist.Dirichlet(struct_params["gamma"] * torch.ones(vocab_size, device=device).unsqueeze(0).expand(struct_upbd["G0"], vocab_size)).to_event(1))
    mixture_components["regression_mu"] = pyro.sample(f"regression_mu", dist.Normal(struct_params["reg_mu_mu"], struct_params["reg_mu_sigma"]).to_event(1))
    mixture_components["regression_sigma"] = pyro.sample(f"regression_sigma", dist.HalfCauchy(struct_params["reg_sigma"]).to_event(1))

    # data fitting
    if (data[0] is not None):
        feature = data[0].to(device)
        N = feature.shape[0]
        M = feature.shape[1]
    else:
        feature = data[0]
        N = 50
        M = 200

    if data[1] is not None:
        label = data[1].to(device)
    else:
        label = data[1]

    assigned_zs = [torch.zeros(N, device=device, dtype=torch.long)]
    with pyro.plate("Data", N):
        for level in range(len(struct_upbd)-1):
            param = cluster_weights[f"L{level}"].unsqueeze(0)[assigned_zs[:]]
            assigned_zs.append(pyro.sample(f"z{level}", dist.Categorical(param)))

        assigned_zs.reverse()
        weights_prior = struct_weights[f"G{len(struct_upbd)-1}"][assigned_zs[:-1]]
        concentrate = struct_params[f"alpha{len(struct_upbd)-1}"][assigned_zs[:-1]].unsqueeze(-1)

        beta = pyro.sample(f"G{len(struct_upbd)}", dist.Beta(torch.ones_like(weights_prior, device=device), weights_prior*concentrate).to_event(1))
        assert beta.shape == (N, struct_upbd["G0"])
        topic_dist = mix_weights(beta)[..., :-1]

        topic_over_docs = topic_dist.unsqueeze(1).expand(-1, M, -1)
        z_gen = pyro.sample(f"z{len(struct_upbd)}", dist.Categorical(probs=topic_over_docs).to_event(1))
        word_dists = mixture_components["generation"][z_gen]  
        obs = pyro.sample("u", dist.Multinomial(1, probs=word_dists).to_event(1), obs=feature)

        z_reg = pyro.sample(f"z{len(struct_upbd)+1}", dist.Categorical(probs=topic_dist))
        reg_mu = mixture_components["regression_mu"][z_reg]
        reg_sigma = mixture_components["regression_sigma"][z_reg]
        reg = pyro.sample("y", dist.Normal(reg_mu, reg_sigma), obs=label)

    return {
        "struct_weights": struct_weights,
        "cluster_weights": cluster_weights,
        "mixture_components": mixture_components,
        "category_assignments": torch.stack(assigned_zs[:-1].reverse(), dim=1),
        "words": {
            "z_gen": z_gen,
            "z_reg": z_reg,
            "obs": obs, 
            "reg": reg
        }
    }

In [39]:
struct_upbd = {"G0": 6, "G1": 2, "G2": 3}
vocab_size = 100
hdmm = model(data=(None, None), struct_upbd=struct_upbd, vocab_size=vocab_size, device="cpu")

In [70]:
print(torch.unique(hdmm["words"]["z_gen"],  return_counts=True))


(tensor([0, 1, 2]), tensor([9993,    4,    3]))


In [ ]:
def suffix_sum(x: torch.Tensor) -> torch.Tensor:
    """
    Compute suffix sums along the last dimension of a tensor.
    Each entry is the sum of all elements to its right.
    The last element along that dimension is always 0.
    
    Example:
        x = torch.tensor([1,2,3])
        suffix_sum(x) -> tensor([5,3,0])
        
        x = torch.tensor([[1,2,3],[4,5,6]])
        suffix_sum(x) -> tensor([[5,3,0],
                                  [11,6,0]])
    """
    # Flip along the last dimension
    rev = torch.flip(x, dims=[-1])
    # Cumulative sum on the flipped tensor
    rev_cumsum = torch.cumsum(rev, dim=-1)
    # Flip back
    suffix = torch.flip(rev_cumsum, dims=[-1])
    # Subtract the original to exclude current element
    suffix = suffix - x
    return suffix


In [ ]:
def group_sum_general(A: torch.Tensor, B: torch.Tensor, return_counts: bool = False, return_labels: bool = False):
    """
    Grouped sum over the last dimension of B based on categories in A.
    Works with arbitrary integer category labels.

    Args:
        A (torch.Tensor): integer tensor of shape (...,) with category labels.
        B (torch.Tensor): tensor of shape (..., D), aligned with A on all but last dim.
        return_counts (bool): if True, also return counts for each category.
        return_labels (bool): if True, also return the unique category labels.

    Returns:
        sums (torch.Tensor): (num_categories, D) grouped sums.
        counts (torch.Tensor, optional): (num_categories,) counts per category.
        labels (torch.Tensor, optional): (num_categories,) original category labels.
    """
    assert A.shape == B.shape[:-1], "A must align with B on all but last dim"

    # flatten all but last dimension
    A_flat = A.reshape(-1)
    B_flat = B.reshape(-1, B.shape[-1])

    # get unique category labels and map them to [0..K-1]
    labels, inv = torch.unique(A_flat, return_inverse=True)
    num_categories = labels.shape[0]
    device = B.device

    sums = torch.zeros((num_categories, B.shape[-1]), device=device, dtype=B.dtype)
    sums.index_add_(0, inv, B_flat)

    results = (sums,)

    if return_counts:
        counts = torch.bincount(inv, minlength=num_categories).to(device)
        results += (counts,)

    if return_labels:
        results += (labels.to(device),)

    if len(results) == 1:
        return results[0]
    return results


In [ ]:
def gibbs_step(level, struct_upbd, model_return, device):
    
    category_summary = {}
    for l in range(len(struct_upbd)-1):
        category_index, category_count = torch.unique(model_return["category_assignments"][:, :l], dim=0, return_counts=True)
        category_summary[f"level_{l}"] = (category_index, category_count)
    components_index, components_count = torch.unique(model_return["words"]["z_gen"],  dim=0, return_counts=True)
    category_summary[f"level_{len(struct_upbd)-1}"] = (components_index, components_count)
    components_index, components_count = torch.unique(model_return["words"]["z_reg"],  dim=0, return_counts=True)
    category_summary[f"level_{len(struct_upbd)}"] = (components_index, components_count)

    data = (model_return["words"]["obs"], model_return["words"]["reg"])
    param_dims = list(struct_upbd.values())
    param_dims.reverse()
    
    param_dims = list(struct_upbd.values())
    param_dims.reverse()
    
    # model parameters
    struct_params = {}
    struct_params["gamma"] = pyro.param("model_gamma", torch.rand(1, device=device), constraint=constraints.positive)
    
    for parent_level in range(len(struct_upbd)-1):  
        child_level = parent_level + 1
        struct_params[f"alpha{parent_level}"] = pyro.param(f"model_alpha{parent_level}", torch.rand(param_dims[-child_level:-1], device=device) , constraint=constraints.positive).unsqueeze(-1).expand(param_dims[-child_level:])
        assert struct_params[f"alpha{parent_level}"].shape == tuple(param_dims[-child_level:])

        struct_params[f"eta{parent_level}"] = pyro.param(f"model_eta{parent_level}", torch.rand(param_dims[-child_level:-1], device=device), constraint=constraints.positive)
        assert struct_params[f"eta{parent_level}"].shape == tuple(param_dims[-child_level:-1])
    struct_params[f"alpha{len(struct_upbd)-1}"] = pyro.param(f"model_alpha{len(struct_upbd)-1}", torch.rand(param_dims[-len(struct_upbd)+1:-1], device=device), constraint=constraints.positive).unsqueeze(-1).expand(param_dims[-len(struct_upbd)+1:])

    struct_params["reg_mu_mu"] = pyro.param("model_reg_mu_mu", torch.zeros([struct_upbd["G0"],], device=device))
    struct_params["reg_mu_sigma"] = pyro.param("model_reg_mu_sigma", torch.ones([struct_upbd["G0"],], device=device), constraint=constraints.positive)
    struct_params["reg_sigma"] = pyro.param("model_reg_sigma", torch.ones([struct_upbd["G0"],], device=device), constraint=constraints.positive)

    # model structures
    struct_weights = {}
    data_summary = category_summary[f"level_{len(struct_upbd)-1}"]
    data_bias = torch.zeros(struct_upbd["G0"], device=device)
    data_bias[data_summary[0]] = data_summary[1]
    alpha_bias = data_bias
    beta_bias = suffix_sum(data_bias)
    param_alpha = torch.ones(1, device=device).expand(param_dims[-1:]) + alpha_bias
    param_beta = struct_params["gamma"].expand(param_dims[-1:]) + beta_bias
    struct_weights["B0"] = (param_alpha, param_beta)
    beta_0 = pyro.sample("G0", dist.Beta(struct_weights["B0"][0], struct_weights["B0"][1]).to_event(1))
    assert beta_0.shape == (param_dims[-1], )

    struct_weights["G0"] = mix_weights(beta_0)[..., :-1]
    assert struct_weights["G0"].shape == (param_dims[-1], )

    for parent_level in range(len(struct_upbd)-1):
        child_level = parent_level + 1
        full_dim = child_level + 1
        data_summary = category_summary[f"level_{parent_level}"]
        data_bias = torch.zeros(struct_upbd[f"G{child_level}"], device=device)
        data_bias[data_summary[0]] = data_summary[1]
        alpha_bias = data_bias
        beta_bias = suffix_sum(data_bias)

        param_alpha = struct_params[f"alpha{parent_level}"]*struct_weights[f"G{parent_level}"].unsqueeze(0).expand(param_dims[-full_dim:]) + alpha_bias
        param_beta = struct_params[f"alpha{parent_level}"]*(1 - struct_weights[f"G{parent_level}"].cumsum(-1)).unsqueeze(0).expand(param_dims[-full_dim:]) + beta_bias
        struct_weights[f"B{child_level}"] = (param_alpha, param_beta)
        beta = pyro.sample(f"G{child_level}", dist.Beta(struct_weights[f"B{child_level}"][0], struct_weights[f"B{child_level}"][1]).to_event(child_level))
        struct_weights[f"G{child_level}"] = mix_weights(beta)[..., :-1]
        assert struct_weights[f"G{child_level}"].shape == tuple(param_dims[-full_dim:])

    mixture_components = {}
    data_summary = category_summary[f"level_{len(struct_upbd)-1}"]
    data_bias = torch.zeros(struct_upbd["G0"], device=device)
    data_bias[data_summary[0]] = data_summary[1]
    gamma = struct_params["gamma"] * torch.ones(vocab_size, device=device).unsqueeze(0).expand(struct_upbd["G0"], vocab_size) + data_bias.unsqueeze(-1)
    mixture_components["generation"] = pyro.sample(f"gen", dist.Dirichlet(gamma).to_event(1))

    # data_summary = category_summary[f"level_{len(struct_upbd)}"]
    # data_bias = torch.zeros(struct_upbd["G0"], device=device)
    # data_bias[data_summary[0]] = data_summary[1]
    mixture_components["regression_mu"] = pyro.sample(f"regression_mu", dist.Normal(struct_params["reg_mu_mu"], struct_params["reg_mu_sigma"]).to_event(1))
    mixture_components["regression_sigma"] = pyro.sample(f"regression_sigma", dist.HalfCauchy(struct_params["reg_sigma"]).to_event(1))

    # data fitting
    if (data[0] is not None):
        feature = data[0].to(device)
        N = feature.shape[0]
        M = feature.shape[1]
    else:
        feature = data[0]
        N = 50
        M = 200

    if data[1] is not None:
        label = data[1].to(device)
    else:
        label = data[1]

    assigned_zs = [torch.zeros(N, device=device, dtype=torch.long)]
    for data_idx in range(N):
        mask = torch.ones(N, device=device, dtype=torch.bool)
        mask[data_idx] = False

        data_point_cat = model_return["category_assignments"][data_idx]
        data_point_word = model_return["words"]["z_gen"][data_idx]
        # Word topic conditional distribution
        words_popu = model_return["words"]["z_gen"][mask][model_return["category_assignments"][mask] == data_point_cat]
        for word_idx in range(M):
            word_mask = torch.ones(M, device=device, dtype=torch.bool)
            word_mask[word_idx] = False
            masked_words = words_popu[:, word_mask]
            topic_index, topic_count = torch.unique(masked_words, return_counts=True)
            topic_prob = torch.zeros(struct_upbd["G0"], device=device)
            for t_idx in topic_index:
                topic_prob[t_idx] = topic_count[t_idx] * dist.Multinomial(1, model_return["mixture_components"]["generation"][t_idx]).log_prob(model_return['words']['obs'][data_idx][word_idx]).exp()
            topic_prob = topic_prob / topic_prob.sum()
            topic_prob = torch.clamp(topic_prob, min=1e-6)
            topic_prob = topic_prob / topic_prob.sum()

        # Regression mixture conditional distribution
        docs_popu = model_return["words"]["z_reg"][mask][model_return["category_assignments"][mask] == data_point_cat]
        reg_index, reg_count = torch.unique(docs_popu, return_counts=True)
        reg_prob = torch.zeros(struct_upbd["G0"], device=device)
        for r_idx in reg_index:
            reg_prob[r_idx] = reg_count[r_idx] * dist.Normal(model_return["mixture_components"]["regression_mu"][r_idx], model_return["mixture_components"]["regression_sigma"][r_idx]).log_prob(model_return['words']['reg'][data_idx]).exp()
        reg_prob = reg_prob / reg_prob.sum()
        reg_prob = torch.clamp(reg_prob, min=1e-6)
        reg_prob = reg_prob / reg_prob.sum()

        category_assignment = {}
        for level in range(len(struct_upbd)-1):
            related_cat = model_return["category_assignments"][mask][model_return["category_assignments"][mask][:, :level-1] == data_point_cat[:level-1]][:, :level+1]
            masked_category_index, masked_category_count = torch.unique(related_cat,  dim=-1, return_counts=True)
            level_prob = torch.zeros(struct_upbd[f"G{level+1}"], device=device)
            for cat_index in masked_category_index:
                level_prob[cat_index[-1]] = masked_category_count[cat_index]
            category_assignment[f"L{level+1}"] = level_prob
        likelihood_term = torch.zeros((struct_upbd["G1"], struct_upbd["G2"]), device=device)
        for s in range(struct_upbd["G1"]):
            for c in range(struct_upbd["G2"]):
                beta_prob = dist.Beta(struct_weights["B2"][0][s, c], struct_weights["B2"][1][s, c]).log_prob(pyro.sample(f"G{len(struct_upbd)}")[data_idx]).exp()
                weights = mix_weights(pyro.sample(f"G{len(struct_upbd)}"))[data_idx]
                words_prob = []
                for m in range(M):
                    topic_prob = dist.Categorical(probs=weights).log_prob(model_return['words']['z_gen'][data_idx][m]).exp()
                    word_prob = dist.Multinomial(1, model_return["mixture_components"]["generation"][model_return['words']['z_gen'][data_idx][m]]).log_prob(model_return['words']['obs'][data_idx][m]).exp() 
                    words_prob.append(topic_prob * word_prob)
                reg_prob = dist.Normal(model_return["mixture_components"]["regression_mu"][model_return['words']['z_reg'][data_idx]], model_return["mixture_components"]["regression_sigma"][model_return['words']['z_reg'][data_idx]]).log_prob(model_return['words']['reg'][data_idx]).exp() * dist.Categorical(probs=weights).log_prob(model_return['words']['z_reg'][data_idx]).exp()
                likelihood_term[s, c] = beta_prob * torch.prod(torch.stack(words_prob)) * reg_prob
        

        assigned_zs.reverse()
        weights_prior = struct_weights[f"G{len(struct_upbd)-1}"][assigned_zs[:-1]]
        concentrate = struct_params[f"alpha{len(struct_upbd)-1}"][assigned_zs[:-1]].unsqueeze(-1)

        beta = pyro.sample(f"G{len(struct_upbd)}", dist.Beta(torch.ones_like(weights_prior, device=device), weights_prior*concentrate).to_event(1))
        assert beta.shape == (N, struct_upbd["G0"])
        topic_dist = mix_weights(beta)[..., :-1]

        topic_over_docs = topic_dist.unsqueeze(1).expand(-1, M, -1)
        z_gen = pyro.sample(f"z_gen{len(struct_upbd)}", dist.Categorical(probs=topic_over_docs).to_event(1))
        word_dists = mixture_components["generation"][z_gen]  
        obs = pyro.sample("u", dist.Multinomial(1, probs=word_dists).to_event(1), obs=feature)

        z_reg = pyro.sample(f"z_reg{len(struct_upbd)}", dist.Categorical(probs=topic_dist))
        reg_mu = mixture_components["regression_mu"][z_reg]
        reg_sigma = mixture_components["regression_sigma"][z_reg]
        reg = pyro.sample("y", dist.Normal(reg_mu, reg_sigma), obs=label)

    return {
        "struct_weights": struct_weights,
        "cluster_weights": cluster_weights,
        "mixture_components": mixture_components,
        "category_assignments": torch.stack(assigned_zs[:-1].reverse(), dim=1),
        "words": {
            "z_gen": z_gen,
            "z_reg": z_reg,
            "obs": obs, 
            "reg": reg
        }
    }

SyntaxError: invalid syntax (3988002856.py, line 2)